# Activity: Should We Trust the Alarm? — FreshKart Cold-Chain Case

**Unit 4 · AI-Assisted Visualization · AI-Assisted Pattern and Anomaly Visualization**
Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## The case

FreshKart's dark stores keep frozen food in **8 freezers** (one per store in Bengaluru, Pune and Hyderabad). Each freezer reports its temperature every hour. Last month FreshKart bought **ColdSense**, an AI service that scores every reading for "anomaly" and raises an alert when the score is **≥ 0.5**.

Cold-chain lead **Ravi** has six weeks of readings, the ColdSense scores, and the **maintenance log** (what technicians actually found). His technicians are tired of being sent out at night. He asks:

> *"Is ColdSense's alert feed trustworthy? Which alerts should my team chase, and what is it missing?"*

**You are the analyst.** Use the data and the visualizations to verify, correct or reject what the vendor claims.

**Ground rules (from the lecture):** an AI score is a *derived attribute*: look at it, don't only threshold it · flags are **candidates, not conclusions** · report precision and recall, not accuracy · scale emphasis to confidence.

**How to work:** run the notebook top to bottom. Complete every cell marked `# TODO` (replace `None` with your code or answer) and write in the cells marked **✏️ Your answer**. Cells that depend on your TODO will tell you if something is still missing.

### The data: `freshkart_coldchain.csv` (synthetic)

| Column | Meaning |
|---|---|
| `timestamp` | hour of the reading (1 Aug – 11 Sep 2026) |
| `city`, `store_id`, `freezer_id` | where the freezer is |
| `temp_c` | freezer temperature (°C); normal is about −18 °C |
| `door_opens` | number of door openings in that hour (busy by day, quiet at night) |
| `compressor_kw` | compressor power draw (kW) |
| `ai_score`, `ai_flag` | ColdSense anomaly score (0–1) and alert (1 if score ≥ 0.5) |
| `confirmed_fault`, `fault_type` | from the maintenance log: 1 if technicians confirmed a real fault in that hour, and what it was |

**Domain note from the maintenance team:** every freezer runs an automatic **defrost cycle from 02:00 to 04:00** each night. The heater warms the coil, so the temperature rises for those two hours. That is normal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (6, 3.2), "axes.spines.top": False, "axes.spines.right": False})
GREY, ACCENT, BLUE = "#bdbdbd", "#d95f02", "#4c78a8"

def ready(*vals):
    """Helper: tells you if a TODO above is still unanswered."""
    ok = all(v is not None for v in vals)
    if not ok:
        print("⚠ Complete the TODO cell(s) above first, then re-run this cell.")
    return ok

df = pd.read_csv("freshkart_coldchain.csv", parse_dates=["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df.head()

## ColdSense's monthly report (to be validated)

| # | Vendor statement |
|---|---|
| 1 | "ColdSense is **97.9% accurate**, so its alerts can be trusted." |
| 2 | "**Most faults happen around 2 am.** We recommend a night technician." |
| 3 | "**All 10 compressor trips** in the period were caught." |
| 4 | "Freezer **FZ-H2 ran without a single fault** in August." |
| 5 | "A reading of **−14 °C is abnormal** for any freezer, at any time." |
| 6 | "An alert with a score of **0.92 means a 92% chance** of a real fault." |
| 7 | "The **default threshold of 0.5** is the best setting for FreshKart." |
| 8 | "A new FZ-B1 reading of **−15.9 °C at 14:00** is **15 standard errors** above normal: a definite fault." |

---
## Part 1 · Data abstraction: items, attributes and derived attributes

Munzner: before choosing a view, ask what the **items** and **attributes** are. The AI does not change the items; it **derives** new attributes.

In [ ]:
derived_cols = ["ai_score", "ai_flag"]
n_alerts = int((df["ai_flag"] == 1).sum())
fault_rate_pct = round(df["confirmed_fault"].mean() * 100, 2)

if ready(derived_cols, n_alerts, fault_rate_pct):
    print(f"derived: {derived_cols} | alerts: {n_alerts} | faults: {df.confirmed_fault.sum()} of {len(df)} rows ({fault_rate_pct}%)")
    display(df[df.confirmed_fault == 1].groupby("fault_type").size().rename("fault hours"))

**✏️ Your answer (Part 1):**

`ai_score` is a continuous quantitative attribute, so position or colour intensity is suitable for showing its magnitude. `ai_flag` is binary/categorical, so a distinct shape or colour can show whether a reading was flagged. The binary flag throws away the score's ranking and confidence information: it does not show how close a reading is to the 0.5 threshold or distinguish a score of 0.51 from 0.99.

---
## Part 2 · Distributions: bin width and the score distribution

### 2a · Temperature histogram

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 5))
for col, w in enumerate([0.1, 1, 8]):
    for row, top in enumerate([None, 15]):
        ax = axes[row, col]
        ax.hist(df["temp_c"], bins=np.arange(-21, 0 + w, w), color=BLUE)
        if top:
            ax.set_ylim(0, top)
            ax.set_title(f"same, y-axis cut at {top} readings", fontsize=9)
        else:
            ax.set_title(f"bin width = {w} °C")
        ax.set_xlabel("temp (°C)")
plt.tight_layout()
plt.show()

warm_tail_visible_bins = [0.1, 1]
n_warm_readings = int((df["temp_c"] > -8).sum())
print("Bin widths that reveal the warm tail separately:", warm_tail_visible_bins)
print("Readings warmer than -8 °C:", n_warm_readings)

### 2b · Look at the AI scores before trusting the cut-off
Wilke: compare groups with overlapping densities, not stacked histograms.

In [ ]:
xs = np.linspace(0, 1, 300)
fig, ax = plt.subplots(figsize=(7, 3))
for lab, col in [(0, GREY), (1, ACCENT)]:
    s = df.loc[df.confirmed_fault == lab, "ai_score"]
    ax.fill_between(xs, stats.gaussian_kde(s)(xs), color=col, alpha=.5, label=f"{'real fault' if lab else 'normal'} (n={len(s)})")
ax.axvline(0.5, color="k", ls="--")
ax.set_yscale("log")
ax.set_ylim(1e-3, 20)
ax.set_xlabel("ai_score")
ax.legend()
ax.set_title("Score density, real faults vs normal readings (log y)")
plt.show()

scores_separate_faults = False
print("Is there a clean score cut-off separating faults from normal readings?", scores_separate_faults)

**✏️ Your answer (Part 2):**

A histogram is a **parameterised algorithm** because its output depends on settings such as bin width and bin boundaries; changing them can reveal or hide patterns in the same data. I would tell my colleague that the 8 °C bins combine some very warm readings with a much larger group, so the anomaly is hidden by over-aggregation. The narrower 0.1 °C and 1 °C views reveal the warm tail more clearly, and the viewer should compare multiple bin widths and inspect the extreme readings rather than conclude that no anomalies exist.

---
## Part 3 · Outlier rules on a short window (vendor statement 3)

Ten consecutive readings from **FZ-P1**, 12 Aug 06:00–15:00. One of them is a compressor trip.

In [ ]:
window = df[(df.freezer_id == "FZ-P1") & (df.timestamp >= "2026-08-12 06:00") & (df.timestamp < "2026-08-12 16:00")]
v = window["temp_c"].values
print(window[["timestamp", "temp_c", "fault_type"]].to_string(index=False))

z_max = float(np.max(np.abs((v - np.mean(v)) / np.std(v, ddof=1))))
q1, q3 = np.percentile(v, [25, 75])
upper_fence = float(q3 + 1.5 * (q3 - q1))

if ready(z_max, upper_fence):
    print(f"largest |z| = {z_max:.2f} -> flagged by |z| > 3? {z_max > 3}")
    print(f"upper IQR fence = {upper_fence:.2f} °C -> trip flagged by IQR rule? {v.max() > upper_fence}")
    print(f"largest |z| possible with n = {len(v)}: {(len(v) - 1) / np.sqrt(len(v)):.2f}")

**✏️ Your answer (Part 3):**

The z-score rule missed the compressor trip because of **masking**: the extreme trip reading pulls the mean and inflates the sample standard deviation, reducing its largest absolute z-score to about 2.83, below the cutoff of 3. The IQR rule is less affected by that one extreme value; its upper fence is about −15.58 °C, so the trip reading at −3.6 °C is clearly above the fence and is flagged.

---
## Part 4 · How good is ColdSense? (vendor statement 1)

In [ ]:
flag = df.ai_flag == 1
real = df.confirmed_fault == 1

TP = int((flag & real).sum())
FP = int((flag & ~real).sum())
FN = int((~flag & real).sum())
TN = int((~flag & ~real).sum())

if ready(TP, FP, FN, TN):
    display(pd.DataFrame([[TP, FN], [FP, TN]], index=["real fault", "normal"], columns=["alert", "no alert"]))

In [ ]:
precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
accuracy = (TP + TN) / len(df)
flag_nothing_accuracy = float((~real).mean())

if ready(precision, recall, f1, accuracy, flag_nothing_accuracy):
    print(f"precision = {precision:.2f} | recall = {recall:.2f} | F1 = {f1:.2f} | accuracy = {accuracy:.1%} | never-alert accuracy = {flag_nothing_accuracy:.1%}")

**✏️ Your answer (Part 4):**

ColdSense has **97.9% accuracy**, but that number is misleading because confirmed faults are rare. At the 0.5 threshold, precision is only **10.1%** (16 of 159 alerts were real), so about **9 in 10 call-outs are false alarms**; recall is **34.8%**, so it catches 16 of 46 confirmed fault-hours and misses **30 of 46**. Ravi should judge the feed using precision and recall, not accuracy alone: the current setting wastes technician time and misses most faults.

---
## Part 5 · Thresholds and costs (vendor statement 7)

A missed fault can spoil a freezer of stock (cost ≈ **40** units); a false call-out costs **1** unit of technician time.

In [ ]:
grid = np.round(np.arange(0.20, 0.96, 0.05), 2)
sweep = pd.DataFrame({"threshold": grid})
sweep["TP"] = [((df.ai_score >= g) & real).sum() for g in grid]
sweep["FP"] = [((df.ai_score >= g) & ~real).sum() for g in grid]
sweep["FN"] = real.sum() - sweep["TP"]
sweep["precision"] = (sweep.TP / (sweep.TP + sweep.FP)).round(2)
sweep["recall"] = (sweep.TP / real.sum()).round(2)

sweep["cost"] = 40 * sweep["FN"] + sweep["FP"]
best_threshold = float(sweep.loc[sweep["cost"].idxmin(), "threshold"])

if ready(best_threshold) and sweep["cost"].notna().all():
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
    ax[0].plot(sweep.threshold, sweep.precision, color=BLUE, label="precision")
    ax[0].plot(sweep.threshold, sweep.recall, color=ACCENT, label="recall")
    ax[0].axvline(0.5, color="k", ls="--", lw=.8)
    ax[0].legend()
    ax[0].set_xlabel("threshold")
    ax[1].plot(sweep.threshold, sweep.cost.astype(float), color=BLUE, marker="o")
    ax[1].axvline(best_threshold, color=ACCENT)
    ax[1].set_xlabel("threshold")
    ax[1].set_ylabel("cost")
    ax[1].set_title(f"lowest cost at {best_threshold:.2f}")
    plt.tight_layout()
    plt.show()
    display(sweep)
    print(f"Lowest estimated cost: {sweep['cost'].min():.0f} at threshold {best_threshold:.2f}")

**✏️ Your answer (Part 5):**

No threshold can fix the detector's main blind spot because all **25 sensor-frozen fault-hours** have very low AI scores (at most 0.096), so none reaches even the lowest threshold tested. The detector scores individual readings, but a frozen sensor looks plausible at each instant (about −18 °C); the fault is the **collective pattern** of an unchanged reading over many hours. Lowering the threshold cannot recover cases whose scores remain near zero, so FreshKart needs a sequence-based or contextual detector as well.

---
## Part 6 · Where do the errors come from?

### 6a · When do the false alarms happen? (vendor statement 2)

In [ ]:
fp_rows = df[flag & ~real]
fp_counts = fp_rows.groupby("hour").size().reindex(range(24), fill_value=0)
fp_counts.plot.bar(color=GREY, title="False alarms by hour of day", figsize=(9, 2.8))
plt.xlabel("hour of day")
plt.ylabel("false alarms")
plt.show()

fp_peak_hour = int(fp_counts.idxmax())
fp_share_at_peak = float(fp_counts.max() / len(fp_rows)) if len(fp_rows) else 0.0

if ready(fp_peak_hour, fp_share_at_peak):
    print(f"{fp_share_at_peak:.0%} of false alarms happen at {fp_peak_hour}:00")

### 6b · Which faults does ColdSense miss?

In [ ]:
caught = df[real].groupby("fault_type")["ai_flag"].agg(caught="sum", fault_hours="size")
caught["recall"] = (caught.caught / caught.fault_hours).round(2)
display(caught)

never_caught_type = str(caught.index[caught["caught"] == 0][0])
anomaly_kind = "collective"
print("Fault type never caught:", never_caught_type)
print("Anomaly kind:", anomaly_kind)

### 6c · Contextual anomalies: compare each reading with its own hour

A **context-aware** baseline compares each reading with the same freezer at the same hour of day (median and MAD over the six weeks).

In [ ]:
grp = df.groupby(["freezer_id", "hour"])["temp_c"]
df["hour_median"] = grp.transform("median")
df["hour_mad"] = grp.transform(lambda s: 1.4826 * stats.median_abs_deviation(s))

mad_safe = df["hour_mad"].replace(0, np.nan)
df["z_context"] = (df["temp_c"] - df["hour_median"]) / mad_safe

if ready(*df["z_context"].head(1)):
    ctx_flag = df["z_context"].astype(float) > 4
    found = df[real & ctx_flag & ~flag]
    print(f"Real faults the context baseline catches but ColdSense missed: {len(found)}")
    display(found[["timestamp", "freezer_id", "temp_c", "hour_median", "z_context", "ai_score", "fault_type"]])
    print(f"False alarms at 02:00 under the context baseline: {(ctx_flag & ~real & (df.hour == 2)).sum()} (ColdSense: {(flag & ~real & (df.hour == 2)).sum()})")

### 6d · Collective anomalies: a run of readings that is abnormal together

In [ ]:
df = df.sort_values(["freezer_id", "timestamp"])
df["roll_sd_6h"] = df.groupby("freezer_id")["temp_c"].transform(lambda s: s.rolling(6).std())

flat = df[df["roll_sd_6h"] < 0.01].copy()

if ready(flat):
    print(flat.groupby("freezer_id")["timestamp"].agg(["min", "max", "count"]))
    h2 = df[(df.freezer_id == "FZ-H2") & (df.timestamp.between("2026-08-19 18:00", "2026-08-21 06:00"))]
    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.plot(h2.timestamp, h2.temp_c, color=GREY)

    # Highlight the entire consecutive flat-reading span for FZ-H2
    flat_h2 = flat[(flat.freezer_id == "FZ-H2") & (flat.timestamp.between("2026-08-19 18:00", "2026-08-21 06:00"))]
    if not flat_h2.empty:
        span_start = flat_h2["timestamp"].min()
        span_end = flat_h2["timestamp"].max() + pd.Timedelta(hours=1)
        ax.axvspan(span_start, span_end, color=ACCENT, alpha=.25, label="6-hour flat sensor span")
        ax.legend(loc="upper left")

    ax.set_title("FZ-H2: is anything unusual here?", loc="left")
    ax.set_ylabel("temperature (°C)")
    plt.show()

df = df.sort_values(["timestamp", "store_id"])

**✏️ Your answer (Part 6):**

**(a)** The 02:00 spike is explained by the scheduled 02:00–04:00 defrost cycle, which warms the coil normally. In this dataset all **143 false alarms** occur at 02:00, so hiring a night technician from this report would create unnecessary call-outs rather than catch the daytime faults, which are concentrated around 15:00–16:00.

**(b)** Vendor statement 5 is not universally true: −14 °C may occur during a normal defrost cycle, while a similar reading at another hour may be a **contextual anomaly** for that freezer. The chart should show the reading against that same freezer's same-hour historical baseline and its normal spread (for example, median and MAD), with the timestamp and defrost window visible.

**(c)** FZ-H2 did have **14 confirmed sensor-frozen fault-hours** in August. A detector that scores readings one at a time sees plausible values near −18 °C; it cannot recognise that the same value has stayed flat for many consecutive hours. That is a **collective anomaly**, so the review must include a time-series or rolling-variability check.

---
## Part 7 · Highlighting, saliency and frequency framing (vendor statement 6)

### 7a · Emphasis scaled to confidence
The vendor's dashboard paints every alert bright red. Implement the lecture's checklist: **≥ 0.90 strong**, **0.70–0.90 moderate**, **0.50–0.70 subtle outline**, **below 0.50 no emphasis** (the data stays visible).

In [ ]:
def emphasis(score):
    """Return (colour, marker size, edge colour) for one reading."""
    if score >= 0.90:
        return ACCENT, 55, "#7f2704"       # strong emphasis
    elif score >= 0.70:
        return "#fdae6b", 35, "#a63603"    # moderate emphasis
    elif score >= 0.50:
        return "none", 26, ACCENT           # subtle outline only
    else:
        return GREY, 12, GREY               # no emphasis

week = df[(df.freezer_id == "FZ-B2") & (df.timestamp.between("2026-08-05", "2026-08-12"))]
if ready(emphasis(0.95)):
    fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    ax[0].plot(week.timestamp, week.temp_c, color=GREY, lw=.8)
    a = week[week.ai_flag == 1]
    ax[0].scatter(a.timestamp, a.temp_c, color="red", s=40)
    ax[0].set_title("Vendor view: every alert in red", loc="left")
    ax[1].plot(week.timestamp, week.temp_c, color=GREY, lw=.8)
    for _, r in week[week.ai_score >= 0.5].iterrows():
        col, size, edge = emphasis(r.ai_score)
        ax[1].scatter(r.timestamp, r.temp_c, color=col, s=size, edgecolors=edge)
    f_ = week[week.confirmed_fault == 1]
    ax[1].scatter(f_.timestamp, f_.temp_c, marker="x", color="k", s=50, label="confirmed fault (log)")
    ax[1].legend(loc="upper left")
    ax[1].set_title("Graded view", loc="left")
    plt.tight_layout()
    plt.show()

### 7b · "A score of 0.92 means 92%"? Count it.
Wilke: people read probabilities better as **frequencies**. Of all alerts in a score band, how many were real?

In [ ]:
band_hi = df[df.ai_score >= 0.9]
band_lo = df[(df.ai_score >= 0.5) & (df.ai_score < 0.7)]

real_per_10_hi = int(round(band_hi["confirmed_fault"].mean() * 10))
real_per_10_lo = int(round(band_lo["confirmed_fault"].mean() * 10))

if ready(real_per_10_hi, real_per_10_lo):
    fig, ax = plt.subplots(2, 1, figsize=(6, 1.8))
    for a_, k, lab in zip(ax, [real_per_10_hi, real_per_10_lo], ["score ≥ 0.9", "score 0.5–0.7"]):
        for i in range(10):
            a_.scatter(i, 0, s=300, color=ACCENT if i < k else "white", edgecolors=ACCENT, linewidths=2)
        a_.axis("off")
        a_.set_title(f"{lab}: of 10 alerts, about {k} were real", loc="left", fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"High-score band: {int(band_hi.confirmed_fault.sum())}/{len(band_hi)} confirmed faults")
    print(f"Mid-score band: {int(band_lo.confirmed_fault.sum())}/{len(band_lo)} confirmed faults")

**✏️ Your answer (Part 7):**

**(a)** Painting every alert bright red gives all alerts the same **salience**, so the cue becomes common and loses its ability to direct attention to the most important cases. Technicians may focus on the red dots and experience **inattentional blindness** toward the unflagged FZ-H2 flat line, where the sensor-frozen fault was missed. Graded emphasis should reserve the strongest colour and size for high-confidence candidates while keeping the raw readings visible.

**(b)** In the supplied data, only **8 of 37** readings with scores ≥0.90 were confirmed faults, or about **2 out of 10** similar high-scoring alerts. A score of 0.92 is therefore not a 92% chance of a real fault; it is a model score, and the observed frequency in the high-score band is much lower.

---
## Part 8 · Standard deviation vs standard error (vendor statement 8)

The vendor judged a new FZ-B1 reading of **−15.9 °C at 14:00** against the 14:00 baseline.

In [ ]:
base = df[(df.freezer_id == "FZ-B1") & (df.hour == 14) & (df.confirmed_fault == 0)]["temp_c"]
mu, sd, n = base.mean(), base.std(), len(base)
new_reading = -15.9

se = sd / np.sqrt(n)
z_sd = (new_reading - mu) / sd
z_se = (new_reading - mu) / se

if ready(se, z_sd, z_se):
    print(f"baseline: mean {mu:.2f} °C, SD {sd:.2f}, n = {n}, SE = {se:.3f}")
    print(f"z using SD = {z_sd:.1f} | z using SE = {z_se:.1f}")

correct_z = "SD"
print("Use", correct_z, "to judge one new reading.")

**✏️ Your answer (Part 8):**

The **standard deviation (SD)** describes how much individual healthy readings vary around their baseline mean. The **standard error (SE)** describes the uncertainty in the estimated baseline mean, not the spread expected for one new reading. For the FZ-B1 14:00 baseline, the new −15.9 °C reading is about **2.3 SD** from the mean; the vendor's value of about 15 comes from incorrectly dividing by the SE, so it is not evidence of a definite fault.

---
## Part 9 · Triage the vendor report

For each statement decide: **Accept** (correct as stated), **Correct** (useful, but the wording or number must change) or **Reject** (not supported). Then measure the report.

In [ ]:
decisions = {
    1: "Correct",  # 97.9% accuracy is close, but it does not establish alert trustworthiness
    2: "Reject",   # 02:00 alarms are normal defrost-related false alarms; real faults peak later
    3: "Accept",   # all 10 compressor trips were flagged
    4: "Reject",   # FZ-H2 had 14 sensor-frozen fault-hours in August
    5: "Reject",   # a -14 °C reading must be judged in time/freezer context
    6: "Reject",   # score is not a calibrated probability
    7: "Reject",   # cost-based sweep favours 0.30, not 0.50
    8: "Reject",   # SD, not SE, describes spread of a single reading
}

if all(v in ("Accept", "Correct", "Reject") for v in decisions.values()):
    accepted = sum(v == "Accept" for v in decisions.values())
    print(f"Statements accepted as written: {accepted}/{len(decisions)} ({accepted / len(decisions):.0%})")
    print(pd.Series(decisions).value_counts().to_string())
else:
    print("⚠ Fill in all eight decisions with 'Accept', 'Correct' or 'Reject'.")

**✏️ Your answer (Part 9):**

I would give Ravi a coordinated anomaly-review view with five parts:

1. **Main view:** a time-series plot of every freezer's raw temperature readings, with alerts shown using graded emphasis rather than painting every alert red.
2. **Context:** each reading compared with that freezer's same-hour median and normal spread, with the scheduled 02:00–04:00 defrost window clearly marked.
3. **Queue:** a ranked list of candidate alerts showing freezer, timestamp, score, likely reason and the relevant time-series segment.
4. **Verification tools:** details-on-demand for the selected reading, plus controls for technicians to confirm or dismiss a candidate and record maintenance findings. Keep the AI layer toggleable so unflagged data remains visible.
5. **Evaluation:** show precision, recall, false alarms, missed faults and results by fault type, alongside the threshold and estimated cost. This makes it clear that the current model catches compressor trips but misses sensor-frozen sequences.

---
## Reflection

**1. Automation bias.**

The 97.9% accuracy figure initially looks convincing because it is precise and sounds authoritative. Without the maintenance log, I might not have noticed that only 16 of 159 alerts were real and that the sensor-frozen fault was missed completely. The maintenance labels are needed to check whether the alarm agrees with what technicians actually found.

**2. Who sets the threshold?**

The threshold should be owned by Ravi and the cold-chain/food-safety operations team, not chosen by the vendor alone. Their dashboard should show the expected false call-outs and missed faults at each threshold, precision, recall, fault-type coverage and the relative cost of spoiled stock versus technician time.

**3. Missing ground truth.**

If faults that nobody inspected remain unlabelled, some true faults will be counted as normal, so the measured precision and recall can be biased and may understate the detector's missed-fault problem. FreshKart can reduce this by auditing a sample of unflagged readings, recording technician outcomes consistently, and feeding confirmed and rejected cases back into periodic model evaluation.

**4. Designing against over-trust.**

I would replace the single red alert flag with graded score emphasis and show the same-hour baseline, recent temperature trace and a short reason for each candidate. A technician should be able to inspect the context and record confirm/dismiss feedback, rather than being told that every alert is definitely a fault.